# 06 - Reading the photon transfer curve

**Purpose.** To explain what session 02 found: what was captured, what `g` is and why a variance
against a signal measures it, what each published number means, and what the project can do now
that it could not do before. `05` is the notebook that *made* these numbers, and is written for
someone checking the work. This one is written for someone deciding what to do next.

**What it is not for.** It measures nothing and writes nothing. Every number is read back from
`results/` - `ptc_rungs.csv`, `ptc_gain.csv`, `ptc_constants.json`, and session 01's
`bias_sweep.csv` and `bias_constants.json`. Where a fit is re-run below it is re-run on the
published rung table, to show what a published number is sensitive to; if any of it disagreed
with `results/`, `results/` would be right and this notebook would be the bug.

**It assumes `00_statistics.ipynb`.** Why noise is measured from a difference of two frames, what
a residual shape means, why a ratio beats a threshold - all of that is explained there, on these
same conventions, and is cited rather than re-derived.

**The headline.** Session 01 measured everything in ADC counts and could not convert any of it.
This session is the conversion: **9.397 electrons per ADC count at gain 0**, falling by a factor
of ten every ~196 gain units. That single number turns every count in the repo into electrons -
and it prices the read-noise cliff session 01 located but could not value.

Three findings do not fit the headline, and they are the ones worth your attention:

1. the gain law **fails its own 1% residual test** (1.34%), so `g` is not interpolable and
   `read_noise_e.csv` was deliberately not written;
2. the FPN test **found a fixed-pattern term** - MISSION's first assumption is refuted rather
   than untested;
3. the archive, which has nothing in common with the bench, agrees with it to **1.3%**.


In [ ]:
import json
import math
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 110, "font.size": 8})

RESULTS = pathlib.Path("..") / "results"
SESSION = pathlib.Path("..") / "data" / "session02"
SPECS = pathlib.Path("..") / "vendor" / "asi585specs" / "gain-curves.csv"

rungs = pd.read_csv(RESULTS / "ptc_rungs.csv")
per_plane = pd.read_csv(RESULTS / "ptc_gain.csv")
with open(RESULTS / "ptc_constants.json") as fh:
    K = json.load(fh)
with open(RESULTS / "bias_constants.json") as fh:
    K1 = json.load(fh)

sweep = pd.read_csv(RESULTS / "bias_sweep.csv")
OFFSET = K1["project_offset"]["value"]
HCG = K1["hcg_threshold_gain"]["value"]
at15 = sweep[sweep.offset == OFFSET].sort_values("gain")

FULL_SCALE = 4095
G = pd.Series({int(k): v for k, v in K["system_gain"]["value"].items()}).sort_index()
G_ERR = pd.Series({int(k): v for k, v in K["system_gain"]["uncertainty"].items()}).sort_index()
LAW = K["gain_law"]["value"]
GAINS = list(G.index)
COLOUR = dict(zip(GAINS, plt.cm.viridis(np.linspace(0, 0.92, len(GAINS)))))

u = rungs[rungs.usable].copy()                 # the rungs the fits actually used
ped = rungs.groupby("gain").pedestal.mean()    # this session's own block-2 pedestal
R = at15.set_index("gain").R_at_offset         # session 01's read noise, in counts

# The session record: the ladder's scale came from gate 3, measured cold.  It is
# the one input below that is not in results/, and section 1 is all it feeds.
HAVE_SESSION = (SESSION / "gate3.csv").exists()
gate3 = pd.read_csv(SESSION / "gate3.csv").set_index("gain") if HAVE_SESSION else None

print(f"{len(rungs)} rungs published, {int(rungs.usable.sum())} used by the fits, "
      f"from {K['system_gain']['source_frames']} frames shot {K['system_gain']['measured_on']}")
print(f"g runs {G.iloc[0]:.4f} down to {G.iloc[-1]:.4f} e- per ADC count over gains "
      f"{GAINS[0]}-{GAINS[-1]}")
if not HAVE_SESSION:
    print("note: data/session02/gate3.csv is absent, so section 1's t_sat panel is skipped.")


---

## 1. What was actually captured

464 frames at -10 C: eight gains, twelve exposures each, four frames per exposure, plus ten
zero-light frames per gain shot minutes away from that gain's ladder.

**The ladder's shape was fixed by the protocol; its scale was not knowable until the session
ran.** Every rung is a percentage of `t_sat(gain)` - the exposure at which the brightest plane
would fill the headroom above its own pedestal - and `t_sat` was *measured* at each gain, cold,
in gate 3. That is deliberate: a saturating exposure extrapolated from a per-sheet attenuation
figure is the kind of number that quietly breaks a whole session, because stacked diffusers do
not stack linearly.

The rungs are **geometric**, 0.3% to 90% at x1.679. L10 is why: with a linearly-spaced ladder
every point sits in the bright end, and a synthetic test recovered **7.1 e- for a true 3.0 e-**
read noise while the gain from the same fit was good to 3%. The five rungs below 4% exist to hold
the low end of the curve down - not because the intercept is the answer, but because a curve that
is unconstrained near zero has a slope that absorbs the error.

**One light level served all eight gains**, which is what makes the right-hand panel below a
result rather than a plot: the exposure that fills the well at gain 450 is 186x shorter than at
gain 0, and that ratio is the gain law showing up in *timings alone*, before any variance is
computed.


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(10.6, 3.0))

for g in GAINS:
    d = rungs[(rungs.gain == g) & (rungs.plane == "R")].sort_values("rung")
    ax[0].plot([g] * len(d), d.exptime_s, "o", ms=3, color=COLOUR[g])
    ax[1].plot(d.pct_of_tsat, d.signal / (FULL_SCALE - ped[g]), "o-", ms=3, lw=0.7,
               color=COLOUR[g], label=f"gain {g}")
ax[0].axhline(1 / 60, color="crimson", lw=0.8, ls="--")
ax[0].text(5, 1 / 55, "one 60 Hz refresh", color="crimson", fontsize=7)
ax[0].set(yscale="log", xlabel="gain", ylabel="exposure, s",
          title="96 exposures: twelve rungs at eight gains")

guide = np.array([0.2, 120.0])
ax[1].plot(guide, guide / 100, color="0.7", lw=0.8, zorder=0)   # signal = commanded fraction
ax[1].set(xscale="log", yscale="log", xlabel="% of measured t_sat",
          ylabel="signal / headroom", title="what actually landed on the sensor")
ax[1].legend(fontsize=6, ncol=2)

if HAVE_SESSION:
    amp = 10 ** (np.array(GAINS) / 200)
    ax[2].plot(GAINS, gate3.t_sat_s, "o-", ms=4, lw=0.8, color="0.3", label="measured, cold")
    ax[2].plot(GAINS, gate3.t_sat_s.iloc[0] / (amp / amp[0]), "--", lw=0.8, color="crimson",
               label="the 0.1 dB law, anchored at gain 0")
    ax[2].set(yscale="log", xlabel="gain", ylabel="t_sat, s",
              title="the gain law, from timings alone")
    ax[2].legend(fontsize=7)
fig.tight_layout()

if HAVE_SESSION:
    print("gate 3: the flux and the saturating exposure, measured cold at every gain")
    print(gate3[["plane", "probe_s", "flux", "t_sat_s", "converged"]].round(4).to_string())
    span = gate3.t_sat_s.iloc[0] / gate3.t_sat_s.iloc[-1]
    print(f"\nt_sat spans {span:.0f}x from gain {GAINS[0]} to {GAINS[-1]}; "
          f"the 0.1 dB law predicts {10 ** (GAINS[-1] / 200):.0f}x "
          f"({span / 10 ** (GAINS[-1] / 200):.3f} of it)")
print(f"\n{int(rungs.sub_refresh.sum() / 4)} of the 96 exposures are shorter than one 60 Hz "
      "refresh, all of them at\ngain 190 and above.  Section 5 returns to what that cost.")


## 2. What a photon transfer curve measures, and why it works

The sensor counts electrons; the file contains ADC counts. `g` is the exchange rate, and nothing
in the file says what it is - the camera does not report how many electrons a count is worth, and
a vendor number for it is a hypothesis.

The trick is that **photons arrive at random**, and randomness has a known shape. If a pixel
collects `N` electrons on average, the number it actually collects varies by `sqrt(N)` - that is
Poisson statistics, and it is not an approximation. Now convert to counts. The signal is
`S = N / g`, and because variance scales with the *square* of a scaling factor, the variance is
`var = N / g^2 = S / g`. So:

```
var(counts) = S(counts) / g + R^2
```

Plot variance against signal, and **the slope is `1 / g`**. Nothing in that line needs the light
source to be uniform, stable or characterised: it needs only that photon arrivals are Poisson and
that the same pixels are measured twice.

That is worth dwelling on, because it is the reason this session could run before the light source
was characterised at all. The curve plots variance against **measured signal**, never against
commanded exposure. A panel that drifts 2% between rungs moves a point *along* the line rather
than off it. L31's 1.79% frame-to-frame instability and L09's 3.8% illumination unevenness are
both real and both irrelevant here - and both would be fatal to the linearity measurement, which
is exactly why that is a different session.

The eight lines below are the whole session in one figure: same sensor, same light, eight
amplifier settings, and the slope changes by a factor of 193 across them.


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.4))

for g in GAINS:
    d = u[(u.gain == g) & (u.plane == "G1")].sort_values("signal")
    ax[0].plot(d.signal, d.var_pair, "o", ms=3, color=COLOUR[g], label=f"gain {g}")
    s = np.logspace(np.log10(d.signal.min()), np.log10(d.signal.max()), 50)
    ax[0].plot(s, s / G[g] + R[g] ** 2, "-", lw=0.8, color=COLOUR[g])
    ax[1].plot(d.signal, d.var_pair / (d.signal / G[g] + R[g] ** 2), "o-", ms=3, lw=0.7,
               color=COLOUR[g])

ax[0].set(xscale="log", yscale="log", xlabel="signal, ADC counts",
          ylabel="pair-difference variance, counts^2",
          title="the photon transfer curve, plane G1\nlines are var = S/g + R^2 with the "
                "published g")
ax[0].legend(fontsize=6, ncol=2)
ax[1].axhline(1, color="0.6", lw=0.8)
ax[1].axhspan(0.97, 1.03, color="0.88", zorder=0)
ax[1].set(xscale="log", xlabel="signal, ADC counts", ylabel="measured / model",
          title="residuals; the band is +/-3%")
fig.tight_layout()

print("the fit is one free parameter: R^2 is passed in, so only the slope is fitted.")
print(f"\nresidual about the fitted line, % rms, per gain and plane:")
print(per_plane.pivot(index="gain", columns="plane", values="resid_pct").round(2).to_string())


## 3. Why a *pair difference*, and why the intercept is not the read noise

Two moves in the analysis do all the work, and both are `00`'s.

**The variance is taken on the difference of two frames, halved.** A single frame's spatial spread
contains everything that varies from pixel to pixel, including anything *fixed* - a pixel that is
reliably 1% more sensitive than its neighbour looks exactly like noise in one frame. Subtract two
frames of the same scene and every fixed feature cancels, leaving only what changed between them:
shot noise and read noise. Halving corrects for having differenced two noisy frames.

That is what makes section 8's test a test. If the variance came from a single frame, "is there a
fixed-pattern term?" would be unanswerable, because the fixed pattern would already be inside the
number.

**The intercept is not read noise, and is not treated as one.** Read noise *is* the intercept in
the algebra - set `S = 0` and `var = R^2` - but a fit's intercept is an extrapolation off the end
of the data, and L10 measured what that costs: a synthetic PTC with a true 3.0 e- read noise
returned **7.1 e-** from its intercept while the slope from the same fit was good to 3%. So `R`
is *passed in* from session 01, where it was measured directly from bias pairs at this gain and
this offset, and the fit has one free parameter instead of two.

The fitted intercept is still computed, and it is a good diagnostic: it should land near the `R`
that was passed in, and where it does not, the fit is telling you something about that gain.


In [ ]:
free = per_plane.groupby("gain").agg(R_fit=("R_fit", "mean"), R_in=("R_counts", "first"),
                                     g=("g", "mean"), g_free=("g_free", "mean"))
free["R_ratio"] = free.R_fit / free.R_in
free["g_shift_pct"] = 100 * (free.g_free / free.g - 1)

fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.0))
ax[0].plot(free.index, free.R_in, "o-", ms=4, lw=0.8, color="0.3", label="passed in (session 01)")
ax[0].plot(free.index, free.R_fit, "s", ms=4, color="crimson", label="this fit's intercept")
ax[0].set(yscale="log", xlabel="gain", ylabel="R, ADC counts",
          title="the intercept as a diagnostic, not an answer")
ax[0].legend(fontsize=7)

ax[1].axhline(1, color="0.6", lw=0.8)
ax[1].axhspan(0.9, 1.1, color="0.88", zorder=0)
ax[1].plot(free.index, free.R_ratio, "o-", ms=4, lw=0.8, color="crimson")
ax[1].set(xlabel="gain", ylabel="fitted / passed in",
          title="ratio; the band is +/-10%")
fig.tight_layout()

print(free.round(4).to_string())
print("\nThe intercept runs high where the low rungs carry variance the model does not "
      "predict --\nand it moves g by less than "
      f"{free.g_shift_pct.abs().max():.1f}% even at its worst, which is the point of "
      "passing R in.")
print("\nThe same frames also gave a block-2 pedestal per gain.  Session 01's fitted pedestal "
      "would\nhave been the alternative, and L14 is why it was not: a dark sitting one count "
      "below a bias\nshot four hours earlier produced a negative dark current in a retired "
      "attempt.")


## 4. `g(gain)`, and the law it is supposed to obey

ZWO's gain control is calibrated in **0.1 dB per unit**. Decibels are logarithmic, so 200 units is
10 dB is a factor of ten in amplification - and since `g` is electrons per count, more
amplification means *fewer* electrons per count. The prediction is exact and has no free
parameters:

```
log10 g = log10 g0 - 0.005 * gain
```

A straight line of slope **-0.00500** in log space. L29 measured -0.00502 on a retired rig and put
unity gain - where one count is one electron, a landmark rather than a target - at gain 194; ZWO
annotate `GAIN=195` on their own published panel.

The measurement below gives **-0.005102**, 2.0% steeper than the law, with unity gain at
**192.6**. The per-gain agreement with L25's retired sweep is inside 0.7% at six of seven shared
gains.

And the fit's residual is **1.34%**, which fails the protocol's own 1% test. Section 5 is that
failure, because it is the most consequential thing in the session.


In [ ]:
x = np.array(GAINS, float)
y = np.log10(G.values)
resid_pct = (10 ** (y - (LAW["slope"] * x + LAW["intercept_log10_g"])) - 1) * 100

fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.2))
ax[0].errorbar(x, G.values, yerr=G_ERR.values, fmt="o", ms=4, color="0.25", capsize=2,
               label="measured (error bars are inside the dots)")
xs = np.linspace(-10, 470, 100)
ax[0].plot(xs, 10 ** (LAW["slope"] * xs + LAW["intercept_log10_g"]), color="crimson", lw=1,
           label=f"fit: slope {LAW['slope']:.5f}")
ax[0].plot(xs, G.iloc[0] * 10 ** (-0.005 * xs), ":", color="steelblue", lw=1,
           label="the 0.1 dB law, anchored at gain 0")
ax[0].axhline(1, color="0.75", lw=0.8)
ax[0].axvline(LAW["unity_gain"], color="0.75", lw=0.8)
ax[0].text(LAW["unity_gain"] + 6, 1.6, f"unity gain {LAW['unity_gain']:.1f}", fontsize=7,
           color="0.4")
ax[0].set(yscale="log", xlabel="gain", ylabel="g, e- per ADC count",
          title="the gain law over three decades")
ax[0].legend(fontsize=7)

ax[1].axhspan(-1, 1, color="0.88", zorder=0)
ax[1].axhline(0, color="0.6", lw=0.8)
ax[1].plot(x, resid_pct, "o-", ms=4, lw=0.8, color="crimson")
ax[1].axvline(HCG, color="steelblue", lw=0.8, ls=":")
for xi, yi in zip(x, resid_pct):
    if abs(yi) > 1.5:
        ax[1].annotate(f"gain {int(xi)}", (xi, yi), xytext=(xi + 12, yi), fontsize=7)
ax[1].set(xlabel="gain", ylabel="residual, % of g",
          title=f"residuals: rms {LAW['residual_pct']:.2f}%, and the band is the 1% rule")
fig.tight_layout()

print(pd.DataFrame({"g": G, "g_err": G_ERR, "law": 10 ** (LAW["slope"] * x
                                                          + LAW["intercept_log10_g"]),
                    "residual_pct": resid_pct}).round(4).to_string())
print(f"\nslope {LAW['slope']:.6f}  =  {LAW['slope'] / -0.005:.4f} x the 0.1 dB law, "
      f"{LAW['slope'] / -0.00502:.4f} x L29")
print(f"unity gain {LAW['unity_gain']:.1f}   (L29 predicted 194, ZWO annotate 195)")


## 5. The failure that matters: `g` is not interpolable

The protocol wrote down what each outcome would mean *before* the data existed, and this is one of
the branches it wrote:

> residual over 1%, or a step at HCG -> interpolation is forbidden and the gain set widens

So `read_noise_e.csv` was **not written**. Session 01 measured `R` in counts at 77 gains; carrying
all of them into electrons needs `g` at gains this session never visited, and that is exactly the
interpolation the residual forbids. `R` in electrons therefore exists at **eight** gains, in
`ptc_gain.csv`, and nowhere else.

It is worth being precise about what failed, because the temptation is to blame the fit. The cell
below re-runs it under every reasonable variation - drop the sub-refresh rungs, drop the top two
rungs, drop each gain in turn - and **nothing rescues it below 1%**. Two points carry the whole
residual: gain 0 at **-2.1%** and gain 300 at **-2.2%**. Remove those two and the other six lie on
the line to **0.33%**, which is better than L29's 1.0%.

That is the shape of a real effect, not of scatter. `00` section 11 is the lesson - read residuals
as a shape - and the shape here is *two outliers against six excellent points*, which says the
problem is at those two gains rather than in the law.

What the two have in common is not obvious, and this notebook will not invent a story for it. What
can be said from the published table: gain 300's per-rung `g` climbs 14% from its faintest rung to
its brightest, and its sub-refresh rungs move its fitted `g` by 1.7% on their own; gain 0's climbs
at the *top* instead, where the brightest rung sits at 89% of headroom. One looks like the light
source, the other looks like the sensor.

**The remedy is the protocol's own**: widen the gain set. A second, cheaper pass - the same
ladder at a handful of intermediate gains, with the bottom rungs kept above one refresh period -
either finds those two points to be local and the law good, or finds a systematic curvature that
no line will fit. Either answer is worth a bench hour, and neither is available from these frames.


In [ ]:
def refit(mask):
    """The same weighted, fixed-intercept fit `05` published, on a subset of rungs."""
    out = {}
    for (g, p), d in rungs[mask].groupby(["gain", "plane"]):
        if len(d) < 3:
            continue
        S, V, R2 = d.signal.values, d.var_pair.values, d.R_counts.iloc[0] ** 2
        w = 1 / V ** 2
        out.setdefault(g, []).append(np.sum(w * S * S) / np.sum(w * S * (V - R2)))
    return pd.Series({g: float(np.mean(v)) for g, v in out.items()}).sort_index()


def law_of(s):
    xx, yy = np.array(s.index, float), np.log10(s.values)
    m, b = np.polyfit(xx, yy, 1)
    r = (10 ** (yy - (m * xx + b)) - 1) * 100
    return m, float((10 ** np.sqrt(np.mean((yy - (m * xx + b)) ** 2)) - 1) * 100), r


variants = {
    "as published": rungs.usable,
    "no sub-refresh rungs": rungs.usable & ~rungs.sub_refresh,
    "no top two rungs": rungs.usable & (rungs.rung < 10),
    "no bottom three rungs": rungs.usable & (rungs.rung > 2),
}
print(f"{'rung selection':>24} {'slope':>10} {'residual':>10}")
for name, mask in variants.items():
    m, rms, _ = law_of(refit(mask))
    print(f"{name:>24} {m:10.6f} {rms:9.2f}%")

print(f"\n{'dropping one gain':>24} {'slope':>10} {'residual':>10}")
for g in GAINS:
    m, rms, _ = law_of(G.drop(g))
    print(f"{('without ' + str(g)):>24} {m:10.6f} {rms:9.2f}%")
m, rms, _ = law_of(G.drop([0, 300]))
print(f"{'without 0 and 300':>24} {m:10.6f} {rms:9.2f}%   <- six points on a line")

print("\nper-rung g at the two suspect gains, mean over planes (the shape of the problem):")
shape = (u[u.gain.isin([0, 250, 300])].pivot_table(index="rung", columns="gain",
                                                   values="g_point"))
print((100 * (shape / shape.median() - 1)).round(1).to_string()
      + "\n(% away from each gain's own median; gain 250 is a well-behaved control)")


## 6. The HCG threshold does not appear in `g` - which was a prediction

Session 01 found a cliff: at gain 200 the sensor switches conversion gain and read noise drops by
a factor of three in a single step of 2 gain units. L30 predicted that this step would **not**
appear in electrons per count, and gave the reasoning - ZWO's own published "full well" and
e-/ADU panels are smooth across the threshold while their read-noise and dynamic-range panels
step, which is only arithmetically possible if the first two are derived from a smooth input.

The measurement agrees. Across the threshold `g` moves by **-0.88%** against a fit scatter of
1.34%: there is no step to see. Conversion gain changes how many volts an electron makes, and the
digitiser is calibrated in volts per count - so the two changes cancel in the ratio, and what is
left is the noise.

This is the strongest kind of confirmation available here, because it was written down first and
because it could have come out the other way in a visible manner.


In [ ]:
near = at15[(at15.gain >= 150) & (at15.gain <= 260)]
fig, ax = plt.subplots(1, 2, figsize=(9.4, 3.0))

ax[0].plot(near.gain, near.R_at_offset, "o-", ms=3.5, lw=0.8, color="0.3")
ax[0].axvline(HCG, color="crimson", lw=0.8, ls="--")
ax[0].set(xlabel="gain", ylabel="R, ADC counts",
          title="session 01: read noise steps at the threshold")

ax[1].axhspan(-LAW["residual_pct"], LAW["residual_pct"], color="0.88", zorder=0)
ax[1].axhline(0, color="0.6", lw=0.8)
ax[1].plot(x, resid_pct, "o-", ms=4, lw=0.8, color="crimson")
ax[1].axvline(HCG, color="steelblue", lw=0.8, ls="--")
ax[1].set(xlim=(150, 320), xlabel="gain", ylabel="residual, % of g",
          title="session 02: g does not\n(band is the fit's own scatter)")
fig.tight_layout()

k = K["hcg_step_in_gain"]
print(f"published hcg_step_in_gain: {k['value']:+.2f}% against a scatter of "
      f"{k['uncertainty']:.2f}%")
print(f"R in counts across the step: {float(R[190]):.3f} -> {float(R[200]):.3f} "
      f"({float(R[200] / R[190]):.2f}x)")
print(f"g across the step:           {G[190]:.4f} -> {G[200]:.4f} "
      f"({G[200] / G[190]:.3f}x, and the law alone predicts "
      f"{10 ** (LAW['slope'] * 10):.3f}x)")


## 7. Pricing the cliff - what session 01 could not finish

Session 01 ended by naming what it could not do: it had located the conversion-gain step but could
not say what it was worth, because "lower read noise in counts" is not a benefit until the counts
are electrons. That is now arithmetic.

| | gain 190 | gain 200 |
|---|---|---|
| read noise, counts | 3.298 | 1.077 |
| `g`, e-/count | 1.046 | 0.922 |
| **read noise, electrons** | **3.45** | **0.99** |

**Ten gain units buy a factor of 3.5 in read noise, for a 12% reduction in well.** That is the
whole trade, and it is why the threshold was worth locating to 2 units rather than 10.

The other two columns below are **derived, not measured**, and the distinction is L30's: the
ADC-limited well is `headroom x g` - arithmetic on the pedestal and the gain - and dynamic range
is that well over the read noise. Neither is a new measurement, and the true full well is not
this number: L28 records the response departing 1% from a straight line at 97.3% of the top code,
so the *linear* well is a little smaller and measuring it is the linearity session's job.

Read the table as an argument about where to observe. Dynamic range peaks at gain 0 and again
just above the threshold; between 190 and 200 it jumps 1.6 stops for free. Below the threshold you
are paying read noise for nothing; far above it you have thrown away the well - at gain 450 there
are 187 electrons of it, which will not hold a star.


In [ ]:
elec = pd.DataFrame({"g": G, "R_counts": R.reindex(GAINS)})
elec["R_e"] = elec.g * elec.R_counts
elec["headroom"] = FULL_SCALE - ped.reindex(GAINS)
elec["well_e"] = elec.headroom * elec.g
elec["DR_stops"] = np.log2(elec.well_e / elec.R_e)

fig, ax = plt.subplots(1, 3, figsize=(10.6, 3.0))
for a, col, t in ((ax[0], "R_e", "read noise, electrons"),
                  (ax[1], "well_e", "ADC-limited well, electrons (derived)"),
                  (ax[2], "DR_stops", "dynamic range, stops (derived)")):
    a.plot(elec.index, elec[col], "o-", ms=4, lw=0.8, color="0.3")
    a.axvline(HCG, color="crimson", lw=0.8, ls="--")
    a.set(xlabel="gain", title=t)
ax[0].set(yscale="log")
ax[1].set(yscale="log")
fig.tight_layout()

print(elec.round(3).to_string())
print(f"\nthe cliff, priced: R {elec.R_e[190]:.2f} -> {elec.R_e[200]:.2f} e- "
      f"({elec.R_e[190] / elec.R_e[200]:.2f}x) for {100 * (1 - elec.well_e[200] / elec.well_e[190]):.0f}% "
      f"of the well, worth {elec.DR_stops[200] - elec.DR_stops[190]:+.2f} stops")
print(f"session 01's published read_noise_at_hcg was "
      f"{K1['read_noise_at_hcg']['value']:.4f} counts; in electrons that is "
      f"{K1['read_noise_at_hcg']['value'] * G[200]:.3f} e-")
print("\nWell and dynamic range are arithmetic on g and the pedestal (L30), not measurements.  "
      "The\nlinear well is smaller than the ADC-limited one (L28) and belongs to the "
      "linearity session.")


## 8. The FPN test: MISSION's first assumption is refuted

Everything in the SNR model rests on one assumption - that a sub-exposure's variance is shot noise
plus read noise **and nothing else**. It matters because of how the two behave when a sub gets
longer: shot noise grows with the signal, read noise is paid once per frame, and that asymmetry is
the entire "how long should a sub be?" question. A third term that scales with the *signal itself*
would not dilute at all, and would put a ceiling on what a longer sub can buy.

The test is the comparison section 3 set up. The pair difference removes fixed pattern; a single
frame keeps it. At the same signal level:

```
var_single - var_pair = (PRNU * S)^2
```

**And the yardstick is not zero.** Two honest estimates of the same variance differ, so the
session shot *two disjoint pairs* at every rung. The excess counts only if it exceeds that
difference - anything smaller is a statement about the estimator, not about the sensor.

It exceeds it by two orders of magnitude. The single-frame variance runs **10.6%** above the pair
variance where two pairs of the same rung differ by **0.28%**. Fitting the excess against `S^2`
gives a PRNU of **1.02%**, against L32's retired 0.61% over a crop.

**What it means, stated carefully.** PRNU is *pixel response non-uniformity*: some pixels are
slightly more sensitive than others, so it is proportional to the signal, and unlike shot noise it
is the **same pattern every frame**. Three consequences follow, and they pull in different
directions:

- it becomes the dominant noise term above `1/PRNU^2` electrons per pixel - about **9,600 e-** -
  which is inside this camera's well at low gain and far outside it at high gain. That is why the
  excess in the table below shrinks with gain: at gain 450 a full pixel holds 187 e-, and 1% of
  that is nothing next to the shot noise;
- because it is fixed, **flat-fielding divides it out**, down to the noise of the flat itself.
  This measurement says the term exists in a *raw* frame. It does not say it survives calibration;
- but it does put a floor on a *flat*'s own quality, and it is the reason a stack of flats stops
  following sqrt(N) - which is how L32 measured it in the first place.

So the honest statement for the model is: the assumption is refuted for raw frames, the term is
characterised, and whether it reaches the final SNR depends on the flat-field step - which is not
yet in the model and now has a reason to be.


In [ ]:
u["excess"] = u.var_single - u.var_pair
u["ratio"] = u.var_single / u.var_pair
u["prnu"] = np.sqrt(u.excess.clip(lower=0)) / u.signal
u["repeat_pct"] = 100 * u.var_pair_spread / u.var_pair
# Below a tenth of headroom the excess is smaller than the estimator's own noise,
# so a PRNU read off there is measuring the estimate rather than the sensor.
bright = u[u.signal >= 0.10 * (FULL_SCALE - u.pedestal)]

fig, ax = plt.subplots(1, 3, figsize=(10.6, 3.2))
for g in GAINS:
    d = bright[(bright.gain == g) & (bright.plane == "G1")]
    ax[0].plot(d.var_pair, d.var_single, "o", ms=3, color=COLOUR[g], label=f"gain {g}")
    ax[1].plot(d.signal ** 2, d.excess.clip(lower=1e-3), "o", ms=3, color=COLOUR[g])
lims = [bright.var_pair.min(), bright.var_single.max()]
ax[0].plot(lims, lims, color="0.6", lw=0.8, zorder=0)
ax[0].set(xscale="log", yscale="log", xlabel="pair-difference variance",
          ylabel="single-frame variance", title="if there were no fixed pattern,\nthese "
                                                "would lie on the grey line")
ax[0].legend(fontsize=6, ncol=2)

s2 = np.array([bright.signal.min() ** 2, bright.signal.max() ** 2])
for p, style in ((K["prnu"]["value"], "-"), (0.0061, "--")):
    ax[1].plot(s2, p ** 2 * s2, style, color="crimson", lw=0.9,
               label=f"PRNU {p:.2%}" + ("" if style == "-" else " (L32)"))
ax[1].set(xscale="log", yscale="log", xlabel="signal^2, counts^2",
          ylabel="single - pair variance", title="the excess is proportional to S^2")
ax[1].legend(fontsize=7)

by_gain = bright.groupby("gain").agg(prnu=("prnu", "median"), ratio=("ratio", "median"),
                                     repeat_pct=("repeat_pct", "median"))
by_gain["excess_pct"] = 100 * (by_gain.ratio - 1)
ax[2].plot(by_gain.index, by_gain.excess_pct, "o-", ms=4, lw=0.8, color="crimson",
           label="single-frame excess")
ax[2].plot(by_gain.index, by_gain.repeat_pct, "s-", ms=4, lw=0.8, color="0.4",
           label="pair-to-pair repeatability")
ax[2].set(yscale="log", xlabel="gain", ylabel="% of the pair variance",
          title="the excess against its own yardstick")
ax[2].legend(fontsize=7)
fig.tight_layout()

print(by_gain.assign(prnu_pct=100 * by_gain.prnu)
      [["prnu_pct", "ratio", "excess_pct", "repeat_pct"]].round(3).to_string())
k = K["prnu"]
crossover = 1 / k["value"] ** 2
holds = elec[elec.well_e > crossover]
print(f"\npublished PRNU {k['value']:.4%} +/- {k['uncertainty']:.4%}, against L32's 0.61%")
print(f"fixed pattern overtakes shot noise above 1/PRNU^2 = {crossover:,.0f} e- per pixel")
print("  the ADC-limited well reaches that only at gain "
      + (f"{int(holds.index.max())} and below" if len(holds) else "no gain in the set"))
print(f"\nfpn_term_present = {K['fpn_term_present']['value']}")


## 9. The archive agrees, and it has nothing in common with the bench

The strongest evidence in the session costs nothing. `g = S / (var_flat - var_bias)` needs one
flat pair and one bias pair, and a year of ordinary imaging has both. So the same quantity was
measured on the historic archive at gain 50 and 252 - different nights, different optics,
different light, no shared calibration, and two subtractions instead of a weighted fit through
twelve rungs.

| gain | archive | bench | agreement |
|---|---|---|---|
| 50 | 5.460 | 5.389 (measured here) | **+1.31%** |
| 252 | 0.496 | 0.497 (via the fitted law) | **-0.33%** |

Two datasets with nothing in common agreeing to about 1% is worth more than either alone, and the
gain-252 row is doing double duty: 252 is not in the bench set, so its comparison runs *through*
the law - and the law that section 5 says is not good enough to publish an interpolation from is,
at that particular gain, right to a third of a per cent. The failure in section 5 is about the
outliers at 0 and 300, not about the middle of the range.

**It is a cross-check and never a source.** No published constant comes from the archive
(`CLAUDE.md`), so this number is recorded beside the bench value and is not averaged into it.

The header's own `EGAIN` is in the same table, and it is now demoted to a standing check: it
disagrees by -1.9% at gain 50 and +3.8% at 252, which is a vendor law good enough to sanity-check
a frame and not good enough to calibrate one.


In [ ]:
arc = K["archive_cross_check"]["value"]
if arc:
    t = pd.DataFrame(arc).T
    t.index.name = "gain"
    t["egain_vs_bench_pct"] = 100 * (t.egain_hdr / t.bench_g - 1)
    print(t.round(4).to_string())
    print(f"\nagreement: " + ", ".join(f"gain {g}: {v['agreement_pct']:+.2f}%"
                                        for g, v in arc.items()))
else:
    print("the cross-check did not run in the pass that wrote ptc_constants.json "
          "(the archive was unreachable);\nthe bench values stand on their own until it does.")


## 10. Against L25 and against ZWO

L25 is a retired project's 61-point PTC on this same camera model, and it was carried into this
repo as a *prediction* - something to reproduce or refute, never to import. The protocol named the
band in advance: `g0` landing in **9.38-9.46** would mean L25, the header `EGAIN` law and ZWO's
`GAIN=195` annotation were all reproduced at once.

`g0 = 9.397`. It lands in the band, and six of the seven shared gains agree inside 0.7%. The
exception is gain 300 at -2.6%, which is the same gain section 5 already flagged - the two
disagreements are one disagreement.

ZWO's own published curve is a different matter and the comparison runs the other way. It is read
off a plot by eye, worth +/-5% at best, and it sits **4-8% above** this measurement over gains
0-300 - one-sided, a systematic offset rather than scatter, which is what a curve drawn for a
product page tends to look like next to a measurement.

The last two rows are worse - 12% at gain 200 and 19% at gain 450 - and that is the read-off,
not the camera. Their chart is quoted there as `1.05` and `0.06`: at two significant figures
the quantisation alone is +/-5% and +/-8%, so those rows cannot resolve a disagreement of the
size being tested. It is not worth chasing either way; it is a reminder of what a vendor number
is worth, which is why the project's rule makes one a hypothesis.


In [ ]:
cmp = pd.DataFrame({"measured": G})
cmp["L25"] = pd.Series({int(k): v for k, v in K["vendor_prediction"]["value"].items()})
cmp["vs_L25_pct"] = 100 * (cmp.measured / cmp.L25 - 1)
if SPECS.exists():
    spec = pd.read_csv(SPECS)
    spec = spec[(spec.branch == "hcg") == (spec.gain >= HCG)].set_index("gain")
    cmp["zwo"] = spec.g_e_per_adu
    cmp["vs_zwo_pct"] = 100 * (cmp.measured / cmp.zwo - 1)

fig, ax = plt.subplots(figsize=(6.6, 3.0))
ax.axhspan(-5, 5, color="0.9", zorder=0)
ax.axhline(0, color="0.6", lw=0.8)
ax.plot(cmp.index, cmp.vs_L25_pct, "o-", ms=4, lw=0.8, color="crimson", label="vs L25")
if "vs_zwo_pct" in cmp:
    ax.plot(cmp.index, cmp.vs_zwo_pct, "s-", ms=4, lw=0.8, color="steelblue",
            label="vs ZWO's chart")
ax.set(xlabel="gain", ylabel="measured - predicted, %",
       title="two predictions meeting the data\n(band is the +/-5% a chart read-off is worth)")
ax.legend(fontsize=7)
fig.tight_layout()

print(cmp.round(4).to_string())
k = K["g_at_gain0"]
print(f"\npublished g_at_gain0 = {k['value']} +/- {k['uncertainty']} {k['unit']}")
print(f"L25 predicted 9.382; measured / predicted = {G[0] / 9.382:.4f}")


## 11. What the session settled, and what it did not

**Settled, and available to every later notebook:**

| constant | value | what it unlocks |
|---|---|---|
| `system_gain` | 9.397 to 0.0486 e-/count over gains 0-450 | every count in the repo becomes electrons |
| `g_at_gain0` | 9.397 +/- 0.032 | the anchor; L25 and `EGAIN` reproduced |
| `gain_law` | slope -0.005102, unity gain 192.6 | the shape of `g(gain)` - **not** licensed for interpolation |
| `hcg_step_in_gain` | -0.88% against 1.34% scatter | L30 confirmed: the cliff is in noise, not in `g` |
| `prnu` | 1.02% +/- 0.45% | a fixed-pattern term the model does not have |
| `archive_cross_check` | +1.3% at gain 50 | the bench and a year of ordinary frames agree |

**And two things that are not constants but are results:**

- **the read-noise cliff is priced**: 3.45 e- -> 0.99 e- across gain 190 to 200, for 12% of the
  well. Gain 200 is now defensible as an observing setting on measured grounds rather than
  folklore;
- **`R` in electrons exists at eight gains**, in `ptc_gain.csv`. That is the row MISSION's
  constants table most wanted, and it arrived narrower than hoped.

**Not settled, and worth being explicit about:**

- **`g` may not be interpolated.** The law's 1.34% residual failed the protocol's test, `g` at an
  unvisited gain is not available, and `read_noise_e.csv` does not exist. The remedy is a wider
  gain set, and the two suspect points - 0 and 300 - are where it should be densest.
- **MISSION's first assumption is refuted.** There is a fixed-pattern term of about 1% of signal.
  What is *not* known is whether it survives flat-fielding, and the model has no flat-field step
  to answer that with. That is a model question, not a bench one.
- **Linearity, full well and `ceiling(gain)` are untouched**, deliberately. Every well figure in
  section 7 is `headroom x g`, arithmetic in L30's sense, and L28's 1%-departure point says the
  linear well is smaller. That measurement needs a characterised light source, a shrunk ROI and a
  per-channel bend - a separate session, and the next one this project has an argument for.
- **Dark current is untouched**, and L14 predicts the dark session yields an upper bound rather
  than a value. `03-dark-bound.md` is written and unrun.

**The LEGACY queue.** L25, L29, L30, L32 and L11 all have verdicts now - two reproductions, one
confirmation, one refutation and one agreement - and L10's ladder argument was consumed in the
design. Moving them out of `LEGACY.md` to their destinations is the step that closes session 02,
and it is a conversation rather than a cell.
